### First code to do a broad multi-model, multi-fold overview (with threshold and costs)

Multi-model, multi-fold (simple sign-based backtest) → find best model type

In [1]:
# Step 1: Move to project root
%cd c:/Users/moham/OneDrive/ml_bot_trading

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt


# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from models.model_training import (
    select_features_rf_reg,
    walk_forward_splits
)
# Updated simulate_trading that handles cost
from backtests.simple_backtest import simulate_trading, calculate_sharpe_ratio

from features.labeling_schemes import calculate_future_returns

# Sklearn / Models
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor



###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=1000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# Prepare X, y
X = df.drop(columns=["future_returns"])
y = df["future_returns"]

###########################################################
# 2) WALK-FORWARD SPLITS
###########################################################
folds = walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds created: {len(folds)}")

###########################################################
# 3) DEFINE MODELS
###########################################################
models = {
    "GradientBoostingRegressor": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42),
    "RandomForestRegressor": RandomForestRegressor(n_estimators=100, random_state=42),
    "LinearRegression": LinearRegression(),
    "SVR": SVR(C=1.0, epsilon=0.2),
    "KNeighborsRegressor": KNeighborsRegressor(n_neighbors=5),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.1),
    "ElasticNet": ElasticNet(alpha=0.1, l1_ratio=0.5),
    "DecisionTreeRegressor": DecisionTreeRegressor(max_depth=5),
    "XGBRegressor": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "LGBMRegressor": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
}

###########################################################
# 4) LOOP OVER FOLDS + SIMPLE BACKTEST (with threshold & cost)
###########################################################
from sklearn.metrics import mean_squared_error

fold_results = {}
threshold = 0.0005  # Only trade if predicted next-bar return > +0.0005 or < -0.0005
cost = 0.0002       # 0.02% transaction cost each position change

for fold_i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n===== Fold {fold_i} =====")

    # Feature selection
    rf_for_fs = RandomForestRegressor(n_estimators=100, random_state=42)
    X_train_sel, selected_idx = select_features_rf_reg(
        X_train_fold, y_train_fold, estimator=rf_for_fs, max_features=20
    )

    # Convert integer positions to actual column names
    feats = X_train_fold.columns[selected_idx]
    print(f"Selected features for Fold {fold_i}: {feats.tolist()}")

    X_test_sel = X_test_fold[feats]

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled = scaler.transform(X_test_sel)

    # We'll store MSE and backtest results for each model
    fold_results[fold_i] = {}

    for model_name, model in models.items():
        # 4.1 Train
        model.fit(X_train_scaled, y_train_fold)
        preds = model.predict(X_test_scaled)

        # 4.2 Evaluate MSE
        mse = mean_squared_error(y_test_fold, preds)

        # 4.3 Convert predictions => signals with threshold
        signals = np.where(preds > threshold, 1, np.where(preds < -threshold, -1, 0))

        # 4.4 Align the test portion of df with the same indices
        df_test_fold = df.loc[X_test_fold.index].copy()

        # 4.5 Run the simple backtest with cost
        daily_returns, total_return = simulate_trading(signals, df_test_fold, cost=cost)
        sr = calculate_sharpe_ratio(np.array(daily_returns))

        fold_results[fold_i][model_name] = {
            "MSE": mse,
            "TotalReturn": total_return,
            "Sharpe": sr
        }

    # End of model loop

# End of fold loop

###########################################################
# 5) PRINT RESULTS
###########################################################
for fold_i, model_dict in fold_results.items():
    print(f"\n=== Fold {fold_i} Results ===")
    for model_name, stats in model_dict.items():
        mse = stats["MSE"]
        ret = stats["TotalReturn"]
        sr = stats["Sharpe"]
        print(f"{model_name}: MSE={mse:.2e}, Return={ret:.2f}%, Sharpe={sr:.2f}")


c:\Users\moham\OneDrive\ml_bot_trading
Number of folds created: 3

===== Fold 1 =====
Selected features for Fold 1: ['trend_dpo', 'trend_kst_diff', 'volume_obv', 'volume_adi', 'trend_mass_index', 'momentum_pvo_signal', 'trend_adx_pos', 'momentum_pvo_hist', 'volume_nvi', 'tick_volume', 'trend_adx', 'momentum_stoch_signal', 'trend_stc', 'trend_adx_neg', 'volume_em', 'momentum_pvo', 'volume_mfi', 'volatility_atr', 'volatility_kcw', 'volatility_bbw']


  File "c:\Users\moham\miniconda3\envs\ml\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\moham\miniconda3\envs\ml\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000216 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1648
[LightGBM] [Info] Number of data points in the train set: 249, number of used features: 20
[LightGBM] [Info] Start training from score 0.000962
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, b

### Insert a hyperparam tuning step after Code 1 to refine the chosen model

(Hyperparam Tuning): Time-based CV (TimeSeriesSplit) on your chosen model → find best hyperparams → save pipeline to “best_rf_pipeline.pkl

In [2]:
# Code 2: Hyperparameter Tuning for Chosen Model

# Step 1: Move to project root
%cd c:/Users/moham/OneDrive/ml_bot_trading

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import joblib

# Sklearn / Models
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import make_scorer, mean_squared_error

# Optional: If you want a pipeline
from sklearn.pipeline import Pipeline

# Your modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import calculate_future_returns

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
# In Code 1, you discovered that RandomForestRegressor generally outperforms.
# Now we do a narrower hyperparam tuning on the older portion of the data.

if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

X_full = df.drop(columns=["future_returns"])
y_full = df["future_returns"]

# Sort df if needed to ensure chronological order
# df = df.sort_index()  # or df = df.sort_values('time') if you have that column
# Then X_full and y_full should match that order

###########################################################
# 2) DEFINE YOUR TRAIN PORTION
###########################################################
# For example, let's use the first 80% for hyperparam tuning
split_idx = int(len(X_full)*0.8)
X_tune = X_full.iloc[:split_idx].copy()
y_tune = y_full.iloc[:split_idx].copy()

print(f"Tuning portion size: {len(X_tune)}")

###########################################################
# 3) TIME-BASED CV (TimeSeriesSplit)
###########################################################
tscv = TimeSeriesSplit(n_splits=3)

# We'll define a negative MSE scorer, because scikit-learn tries to maximize the score
def neg_mse_scorer(estimator, X, y):
    preds = estimator.predict(X)
    return -mean_squared_error(y, preds)

scorer = make_scorer(mean_squared_error, greater_is_better=False)

###########################################################
# 4) BUILD A PIPELINE (optional) OR JUST USE RF
###########################################################
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("rf", RandomForestRegressor(random_state=42))
])

###########################################################
# 5) DEFINE PARAM DISTRIBUTIONS FOR RandomForest
###########################################################
param_distributions = {
    "rf__n_estimators": [100, 200, 300],
    "rf__max_depth": [None, 5, 10, 15],
    "rf__min_samples_split": [2, 5, 10],
    "rf__max_features": ["auto", "sqrt", 0.5]
}

###########################################################
# 6) SET UP RandomizedSearchCV
###########################################################
from sklearn.model_selection import RandomizedSearchCV

random_search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=10,             # how many random combos
    scoring=neg_mse_scorer,  # or you can use 'neg_mean_squared_error'
    cv=tscv,               # time-based folds
    random_state=42,
    n_jobs=-1,
    verbose=2
)

###########################################################
# 7) FIT ON TUNING PORTION
###########################################################
random_search.fit(X_tune, y_tune)

print("Best params:", random_search.best_params_)
print("Best neg MSE:", random_search.best_score_)

best_estimator = random_search.best_estimator_

###########################################################
# 8) SAVE THE BEST ESTIMATOR
###########################################################
joblib.dump(best_estimator, "best_rf_pipeline.pkl")
print("Saved best estimator to 'best_rf_pipeline.pkl'")


c:\Users\moham\OneDrive\ml_bot_trading
Tuning portion size: 1599
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best params: {'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__max_features': 0.5, 'rf__max_depth': 5}
Best neg MSE: -0.00014566021556092866
Saved best estimator to 'best_rf_pipeline.pkl'


### Second code for a deeper vectorbt backtest on the best models or an expanding walk-forward approach to confirm their robustness

Final approach with vectorbt in an expanding walk-forward manner (or single time-based split) for a more in-depth backtes

(This snippet): Load “best_rf_pipeline.pkl”, do final expanding walk-forward with vectorbt, threshold, and fees → get final robust out-of-sample performance

In [3]:
# THIRD CODE: Final Expanding Walk-Forward with VectorBT
# using the best pipeline loaded from best_rf_pipeline.pkl

# Step 1: Move to project root
%cd c:/Users/moham/OneDrive/ml_bot_trading

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt
import joblib

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features
from features.labeling_schemes import calculate_future_returns

# Sklearn
from sklearn.metrics import mean_squared_error

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# If df's index is date/time, ensure it's sorted chronologically if needed:
# df = df.sort_index()

X = df.drop(columns=["future_returns"])
y = df["future_returns"]

###########################################################
# 2) DEFINING EXPANDING SPLITS
###########################################################
def expanding_walk_forward_splits(X, y, n_splits=3):
    """
    Creates expanding walk-forward folds. For each fold i:
      - Train: [0 : fold_i]
      - Test:  [fold_i : fold_{i+1}]
    The last fold extends to the end.
    """
    n = len(X)
    fold_size = n // (n_splits + 1)
    splits = []
    
    for i in range(n_splits):
        train_end = (i+1) * fold_size
        if i == n_splits - 1:
            test_end = n
        else:
            test_end = (i+2) * fold_size
            if test_end > n:
                test_end = n
        
        X_train_fold = X.iloc[:train_end]
        y_train_fold = y.iloc[:train_end]
        
        X_test_fold = X.iloc[train_end:test_end]
        y_test_fold = y.iloc[train_end:test_end]
        
        splits.append((X_train_fold, y_train_fold, X_test_fold, y_test_fold))
    return splits

folds = expanding_walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds: {len(folds)}")

###########################################################
# 3) LOAD BEST PIPELINE & RUN EXPANDING WALK-FORWARD
###########################################################
# This pipeline was saved in the second code (hyperparam tuning step)
best_pipeline = joblib.load("best_rf_pipeline.pkl")
print("Loaded best pipeline from 'best_rf_pipeline.pkl'")

threshold = 0.0005  # min predicted return for a trade
fees = 0.0002       # 0.02% transaction cost per trade

fold_results = {}

for i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n=== Fold {i} ===")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # 3.1 Fit the pipeline on this fold's train portion
    # The pipeline includes: [("scaler", StandardScaler()), ("rf", RandomForestRegressor with best hyperparams)]
    best_pipeline.fit(X_train_fold, y_train_fold)
    
    # 3.2 Predict on the test portion
    preds = best_pipeline.predict(X_test_fold)
    mse = mean_squared_error(y_test_fold, preds)
    
    # 3.3 Convert predictions => signals with threshold
    signals = np.where(preds > threshold, 1, np.where(preds < -threshold, -1, 0))
    
    # 3.4 Vectorbt backtest on the test portion
    df_test_fold = df.loc[X_test_fold.index].copy()  # label-based indexing
    close_prices = df_test_fold["close"]
    
    # pad signals if needed
    if len(signals) < len(close_prices):
        signals = np.append(signals, [0]*(len(close_prices)-len(signals)))
    
    signals_s = pd.Series(signals, index=close_prices.index)
    
    pf = vbt.Portfolio.from_signals(
        close_prices,
        entries=signals_s > 0,
        exits=signals_s < 0,
        init_cash=10000,
        freq='4H',
        fees=fees
    )
    
    total_return = pf.total_return()
    sharpe_ratio = pf.sharpe_ratio()
    
    print(f"Fold {i} MSE={mse:.2e}, Return={total_return:.2f}%, Sharpe={sharpe_ratio:.2f}")
    
    # Full stats
    print("\nVectorbt stats for Fold", i)
    print(pf.stats())

    # Optional: Plot
    fig = pf.plot()
    fig.show()
    
    # store results
    fold_results[i] = {
        "MSE": mse,
        "TotalReturn": total_return,
        "Sharpe": sharpe_ratio
    }

###########################################################
# 4) PRINT FOLD RESULTS
###########################################################
for i, stats in fold_results.items():
    print(f"\nFold {i} => MSE={stats['MSE']:.2e}, Return={stats['TotalReturn']:.2f}%, Sharpe={stats['Sharpe']:.2f}")


c:\Users\moham\OneDrive\ml_bot_trading
Number of folds: 3
Loaded best pipeline from 'best_rf_pipeline.pkl'

=== Fold 1 ===
Train size: 499, Test size: 499
Fold 1 MSE=1.75e-04, Return=-0.09%, Sharpe=-0.47

Vectorbt stats for Fold 1
Start                         2024-06-16 12:00:00
End                           2024-09-07 12:00:00
Period                           83 days 04:00:00
Start Value                               10000.0
End Value                             9093.817872
Total Return [%]                        -9.061821
Benchmark Return [%]                   -18.037356
Max Gross Exposure [%]                      100.0
Total Fees Paid                         33.970942
Max Drawdown [%]                        19.898861
Max Drawdown Duration            49 days 08:00:00
Total Trades                                    9
Total Closed Trades                             8
Total Open Trades                               1
Open Trade PnL                        -963.150266
Win Rate [%]       


=== Fold 2 ===
Train size: 998, Test size: 499
Fold 2 MSE=1.08e-04, Return=0.05%, Sharpe=2.42

Vectorbt stats for Fold 2
Start                         2024-09-07 16:00:00
End                           2024-11-29 16:00:00
Period                           83 days 04:00:00
Start Value                               10000.0
End Value                            10483.759265
Total Return [%]                         4.837593
Benchmark Return [%]                     78.30179
Max Gross Exposure [%]                      100.0
Total Fees Paid                         12.487318
Max Drawdown [%]                         2.179345
Max Drawdown Duration            79 days 20:00:00
Total Trades                                    3
Total Closed Trades                             3
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            66.666667
Best Trade [%]                           5.721863
Worst Trade [%]             


=== Fold 3 ===
Train size: 1497, Test size: 502
Fold 3 MSE=1.20e-04, Return=0.10%, Sharpe=1.16

Vectorbt stats for Fold 3
Start                               2024-11-29 20:00:00
End                                 2025-02-21 08:00:00
Period                                 83 days 16:00:00
Start Value                                     10000.0
End Value                                  11022.125418
Total Return [%]                              10.221254
Benchmark Return [%]                           0.960283
Max Gross Exposure [%]                            100.0
Total Fees Paid                               45.155588
Max Drawdown [%]                              13.157389
Max Drawdown Duration                  30 days 12:00:00
Total Trades                                         12
Total Closed Trades                                  11
Total Open Trades                                     1
Open Trade PnL                               565.237117
Win Rate [%]                         


Fold 1 => MSE=1.75e-04, Return=-0.09%, Sharpe=-0.47

Fold 2 => MSE=1.08e-04, Return=0.05%, Sharpe=2.42

Fold 3 => MSE=1.20e-04, Return=0.10%, Sharpe=1.16


Final Training & Save Model

In [4]:
import joblib

# Suppose best_pipeline is your pipeline with best hyperparams (scaler + random forest, etc.)
# Option A: Train on the entire dataset
best_pipeline.fit(X, y)  # where X, y are your entire historical dataset

# Option B: Train on [0 : last_day-1] if you want to keep the last day as a final test

# Now save the final trained pipeline
import joblib

joblib.dump(best_pipeline, "saved_models/final_production_pipeline.pkl")

print("Final production model saved to 'final_production_pipeline.pkl'")


Final production model saved to 'final_production_pipeline.pkl'


### Second code for a deeper vectorbt backtest on the best models or an expanding walk-forward approach to confirm their robustness - Manually or standard hypterparameters

In [ ]:
# Step 1: Move to project root
%cd c:/Users/moham/OneDrive/ml_bot_trading

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import MetaTrader5 as mt5
import vectorbt as vbt

# Our modules
from data.data_loader import get_data_mt5
from features.feature_engineering import add_all_ta_features, calculate_future_returns
from models.model_training import select_features_rf_reg

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

###########################################################
# 1) DATA LOADING & FEATURE ENGINEERING
###########################################################
if not mt5.initialize():
    print("Failed to initialize MT5")
else:
    data = get_data_mt5(symbol="BTCUSD", timeframe=mt5.TIMEFRAME_H4, n_bars=2000)
    mt5.shutdown()

df = add_all_ta_features(data)
df = calculate_future_returns(df).dropna(subset=["future_returns"])

# If df's index is date/time, ensure X and y share that index
# e.g., df = df.sort_index() if needed

X = df.drop(columns=["future_returns"])
y = df["future_returns"]

###########################################################
# 2) DEFINING EXPANDING SPLITS
###########################################################
def expanding_walk_forward_splits(X, y, n_splits=3):
    """
    Creates expanding walk-forward folds. For each fold i:
      - Train: [0 : fold_i]
      - Test:  [fold_i : fold_{i+1}]
    The last fold extends to the end.
    """
    n = len(X)
    fold_size = n // (n_splits + 1)
    splits = []
    
    for i in range(n_splits):
        train_end = (i+1) * fold_size
        if i == n_splits - 1:
            test_end = n
        else:
            test_end = (i+2) * fold_size
            if test_end > n:
                test_end = n
        
        X_train_fold = X.iloc[:train_end]
        y_train_fold = y.iloc[:train_end]
        
        X_test_fold = X.iloc[train_end:test_end]
        y_test_fold = y.iloc[train_end:test_end]
        
        splits.append((X_train_fold, y_train_fold, X_test_fold, y_test_fold))
    return splits

folds = expanding_walk_forward_splits(X, y, n_splits=3)
print(f"Number of folds: {len(folds)}")

###########################################################
# 3) EXPANDING WALK-FORWARD + VECTORBT
###########################################################
model = RandomForestRegressor(n_estimators=100, random_state=42)

threshold = 0.0005  # min predicted return for a trade
fees = 0.0002       # 0.02% transaction cost per trade

fold_results = {}

for i, (X_train_fold, y_train_fold, X_test_fold, y_test_fold) in enumerate(folds, start=1):
    print(f"\n=== Fold {i} ===")
    print(f"Train size: {len(X_train_fold)}, Test size: {len(X_test_fold)}")
    
    # 3.1 Feature selection
    fs_rf = RandomForestRegressor(n_estimators=100, random_state=42)
    X_train_sel, idx_sel = select_features_rf_reg(X_train_fold, y_train_fold, estimator=fs_rf, max_features=20)
    feats = X_train_fold.columns[idx_sel]
    print("Selected features:", feats.tolist())
    
    X_test_sel = X_test_fold[feats]
    
    # 3.2 Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_sel)
    X_test_scaled  = scaler.transform(X_test_sel)
    
    # 3.3 Train final model for this fold
    model.fit(X_train_scaled, y_train_fold)
    
    # 3.4 Predict on test fold
    preds = model.predict(X_test_scaled)
    mse = mean_squared_error(y_test_fold, preds)
    
    # 3.5 Convert predictions => signals with threshold
    signals = np.where(preds > threshold, 1, np.where(preds < -threshold, -1, 0))
    
    # 3.6 Vectorbt backtest on test fold
    # IMPORTANT FIX: use .loc[...] instead of .iloc[...] 
    # to match label-based indices (especially if they're date/time).
    df_test_fold = df.loc[X_test_fold.index].copy()  # FIX HERE
    close_prices = df_test_fold["close"]
    
    if len(signals) < len(close_prices):
        signals = np.append(signals, [0]*(len(close_prices)-len(signals)))
    
    signals_s = pd.Series(signals, index=close_prices.index)
    
    pf = vbt.Portfolio.from_signals(
        close_prices,
        entries=signals_s > 0,
        exits=signals_s < 0,
        init_cash=10000,
        freq='4H',
        fees=fees
    )
    
    total_return = pf.total_return()
    sharpe_ratio = pf.sharpe_ratio()
    
    print(f"Fold {i} MSE={mse:.2e}, Return={total_return:.2f}%, Sharpe={sharpe_ratio:.2f}")
    
    # 3.7 Print full stats
    print("\nVectorbt stats for Fold", i)
    print(pf.stats())

    # 3.8 Plot (optional)
    fig = pf.plot()
    fig.show()
    
    # store results
    fold_results[i] = {
        "MSE": mse,
        "TotalReturn": total_return,
        "Sharpe": sharpe_ratio
    }

###########################################################
# 4) PRINT FOLD RESULTS
###########################################################
for i, stats in fold_results.items():
    print(f"\nFold {i} => MSE={stats['MSE']:.2e}, Return={stats['TotalReturn']:.2f}%, Sharpe={stats['Sharpe']:.2f}")
